# Demo 2: Controlled Pendulum
## Demo 2.2: PyFMI Co-Simulation

### Description

The following demo implements a controlled pendulum system described in Demo 2.1. The Modelica models for the `Reference`, `Pendulum`, `AngleEncoder`, `Controller`, and `Drive` are exported as an Co-Simulation FMU using OpenModelica. The FMUs are then imported and simulated in Python using the PyFMI package.

OpenModelica allows only the FMI 2.0 standard for Co-Simulation FMUs, which is supported by PyFMI.

### Procedure

**1. Changing the working dirctory to use `SysSimX` package**

In [2]:
import os
import sys
from path import Path
repo_root = Path.getcwd().parent.parent
sys.path.insert(0, str(repo_root))

from SysSimX.utilities.update_fmus import get_fmu_paths

**2. Get the FMU files**

- Definition of the directory where the OpenModelica package is located
- Definition of the path where the FMUs should be stored

1. OpenModelica models are imported as ModelicaSystems using `OMPython`.
2. OpenModelica models are built and exported as Co-Simulation FMUs using `OMPython`.
3. The FMUs are copied to the specified path. 

In [3]:
demo_dir_path = Path(repo_root / 'demos' / 'ControlledPendulum')
package_path = Path(demo_dir_path / 'ControlledPendulum')
fmu_output_dir = Path(demo_dir_path / 'FMUs')

fmu_paths = get_fmu_paths(package_path, fmu_output_dir, force_rebuild=False)
for fmu, path in fmu_paths.items():
    print(f"{fmu:<25}: {path}")

All FMUs for package 'ControlledPendulum' already exist. Skipping generation.
AngleEncoder             : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/AngleEncoder.fmu
Demo_Driven              : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Demo_Driven.fmu
Demo_DrivenWithWall      : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Demo_DrivenWithWall.fmu
Demo_UndrivenWallDiscrete: /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Demo_UndrivenWallDiscrete.fmu
Demo_UndrivenWithWall    : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Demo_UndrivenWithWall.fmu
Drive                    : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/Drive.fmu
ImpactWall               : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/ImpactWall.fmu
PID_Continuous           : /home/flo/repos/SystemSimulation/demos/ControlledPendulum/FMUs/PID_Continuous.fmu
PID_Sampled              : /home/flo/repos/Sy

**3. The FMUs are imported using `PyFMI` and initialized**

In [4]:
from pyfmi import load_fmu

ref_fmu = load_fmu(fmu_paths["Reference"])
sensor_ref_fmu = load_fmu(fmu_paths["AngleEncoder"])
sensor_state_fmu = load_fmu(fmu_paths["AngleEncoder"])
pid_fmu = load_fmu(fmu_paths["PID_Continuous"])
drive_fmu = load_fmu(fmu_paths["Drive"])
pendulum_fmu = load_fmu(fmu_paths["Pendulum"])

fmu_list = [ref_fmu, sensor_ref_fmu, sensor_state_fmu, pid_fmu, drive_fmu, pendulum_fmu]

t = 0.0
dt = 0.001
tf = 10.0
h = dt  # step size

for fmu in fmu_list:
    fmu.reset()
    fmu.setup_experiment(start_time=t)
    fmu.initialize()

**4. Co-Simulation Loop**

- FMUs are simulated in a loop with a fixed communication step size.
- Data between FMUs is exchanged at each communication step using getter and setter methods


**PyFMI Features:**
- Importing and simulating FMUs as Co-Simulation units or Model Exchange units.
- Setting and getting variable values using the string names of the variables
- Arguments for setting and getting variables must be **iterables** (list, tuple, np.array)
- Co-Simulation FMUs have a `doStep` method for advancing the simulation by a specified step size.

In [5]:
# Initialize logging arrays
ts = []
q_ref_log, q_state_log, omega_state_log = [], [], []
U_ref_log, U_state_log = [], []

while t < tf - 1e-9:
    # 1) Read the current outputs
    ref_fmu.do_step(current_t=t, step_size=h)
    q_ref = float(ref_fmu.get("q_ref")[0])

    # 2) Read plant state at current time
    q_state = float(pendulum_fmu.get("q_state")[0])
    omega_state = float(pendulum_fmu.get("omega_state")[0])

    # 3) Sensors: set inputs -> step -> read outputs
    sensor_ref_fmu.set("q", q_ref)
    sensor_state_fmu.set("q", q_state)
    sensor_ref_fmu.do_step(current_t=t, step_size=h)
    sensor_state_fmu.do_step(current_t=t, step_size=h)
    U_ref = float(sensor_ref_fmu.get("U_q")[0])
    U_state = float(sensor_state_fmu.get("U_q")[0])

    error = U_ref - U_state

    # 4) PID: useses sensor voltages as inputs
    pid_fmu.set("y", U_state)
    pid_fmu.set("ref", U_ref)
    pid_fmu.do_step(current_t=t, step_size=h)
    u_control = float(pid_fmu.get("u")[0])

    # 5) Drive
    drive_fmu.set("u_control", u_control)
    drive_fmu.set("omega", omega_state)
    drive_fmu.do_step(current_t=t, step_size=h)
    torque = float(drive_fmu.get("torque")[0])

    # 6) Pendulum
    pendulum_fmu.set("torque", torque)
    pendulum_fmu.do_step(current_t=t, step_size=h)

    # Logging
    ts.append(t)
    q_ref_log.append(q_ref)
    q_state_log.append(q_state)
    omega_state_log.append(omega_state)
    U_ref_log.append(U_ref)
    U_state_log.append(U_state)

    # Increase time
    t += h

**5. Plotting the results**

In [6]:
# Create a simple plotly figure for the q_ref and q_state over time
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts, y=q_ref_log, mode='lines',
                            name='Reference Angle (q_ref)', line=dict(color='white', dash='dash')))
fig.add_trace(go.Scatter(x=ts, y=q_state_log, mode='lines',
                            name='State Angle (q_state)', line=dict(color='red')))
fig.update_layout(title='Pendulum Angle Tracking',
                  xaxis_title='Time (s)',
                  yaxis_title='Angle (degrees)',
                  legend_title='Legend',
                  template='plotly_dark')
fig.show()